# 7.4 - AI Chat with FastAPI

**Module 7 - LLM API Engineering** | Netsetos GenAI Engineering

This notebook works through AI Chat with FastAPI hands-on: The bug that shipped: a bot with no memory; Give every user their own conversation history; Build system prompts you can reuse; Keep the conversation inside a token budget; Make sessions survive a restart with Redis; Build the LLM client once, inject it everywhere.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

A single reproducibility note. The lesson leans on fixed, hand-written example values (session IDs, token counts) rather than live randomness, so there is nothing to seed here - it's a marker that every printed number below is deterministic and repeatable, not a fresh dice roll.

In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

This is a comment-only prep cell, not runnable logic. It exists to set your expectation: the outputs you see are the same ones the lesson author saw.

**How the code works - step by step**
- **The comment** - flags that all example values in the notebook are stable, so your run should match the printed outputs.

**In one line:** nothing executes - it just promises reproducible numbers.

**What you'll see:** (no output - environment prep)

## 1 - The bug that shipped: a bot with no memory

Before any code, look at the failure that motivates the whole lesson. A launch-day demo bot answered a follow-up question with "Which one of WHAT?" because a single backend line sent only the newest message to the model. The team had tested 400 single questions but never a real conversation.

In [ ]:
# Output: the launch demo, reconstructed from the meeting notes
#
# CEO:  "My order #4521 arrived with a cracked screen."
# Bot:  "Sorry to hear that! For damaged items we offer replacement or
#        refund. Could you share a photo of the damage?"
# CEO:  "OK. So which one should I pick?"
# Bot:  "Which one of WHAT? Hi! How can I help you today?"
#
# ...silence in the room.
#
# Post-mortem, one line of backend code:
#   answer = llm(new_message)          # sends ONLY the newest message
# The team had tested 400 single questions. Nobody tested a CONVERSATION.

**Explanation**

This is a narrative comment block, not executable code - running it produces nothing. It reconstructs the post-mortem so the one-line root cause stays vivid: `answer = llm(new_message)` discards all prior turns, so the model has no idea what "which one" refers to.

**How the code works - step by step**
- **The transcript** - a customer reports a cracked screen, the bot offers replacement-or-refund, then blanks on the follow-up.
- **The post-mortem line** - `answer = llm(new_message)` sends ONLY the latest message; there is no history in the call.
- **The test gap** - 400 isolated questions passed; nobody ran a multi-turn conversation.

**In one line:** stateless bots forget everything the instant the next message arrives.

**What you'll see:** No output - it's a story told in comments. Running the cell does nothing; it frames the problem the rest of the notebook solves.

## 2 - Give every user their own conversation history

The fix for the amnesia bug starts here: store each conversation as an ordered list of messages, in the exact `{role, content}` shape LLM APIs expect. This cell builds the data model - a `Message`, a `ChatSession`, and a `SessionStore` that keeps one session per user.

In [ ]:
# =============================================
# CHAT SESSION MANAGER
# Each user gets their own conversation history.
# Messages are stored in the format the LLM expects.
# =============================================

import uuid
from datetime import datetime
from dataclasses import dataclass, field

@dataclass
class Message:
    role: str           # "user", "assistant", or "system"
    content: str
    timestamp: float = field(default_factory=lambda: datetime.now().timestamp())
    tokens: int = 0

class ChatSession:
    """One conversation between one user and the AI."""
    
    def __init__(self, session_id: str = None, system_prompt: str = ""):
        self.session_id = session_id or str(uuid.uuid4())[:8]
        self.messages: list[Message] = []
        self.created_at = datetime.now()
        
        if system_prompt:
            self.messages.append(Message(role="system", content=system_prompt))
    
    def add_user_message(self, content: str):
        self.messages.append(Message(role="user", content=content))
    
    def add_assistant_message(self, content: str):
        self.messages.append(Message(role="assistant", content=content))
    
    def to_api_format(self) -> list[dict]:
        """Convert to the format LLM APIs expect."""
        return [{"role": m.role, "content": m.content} for m in self.messages]
    
    def turn_count(self) -> int:
        return sum(1 for m in self.messages if m.role == "user")

class SessionStore:
    """Manage multiple chat sessions (one per user/conversation)."""
    
    def __init__(self):
        self.sessions: dict[str, ChatSession] = {}
    
    def create(self, system_prompt: str = "") -> ChatSession:
        session = ChatSession(system_prompt=system_prompt)
        self.sessions[session.session_id] = session
        return session
    
    def get(self, session_id: str) -> ChatSession | None:
        return self.sessions.get(session_id)
    
    def delete(self, session_id: str):
        self.sessions.pop(session_id, None)

# ── Demo ──
store = SessionStore()
session = store.create(system_prompt="You are a helpful AI tutor for the Netsetos course.")

session.add_user_message("What is RAG?")
session.add_assistant_message("RAG stands for Retrieval-Augmented Generation...")
session.add_user_message("How does the retrieval part work?")

print(f"Session: {session.session_id}")
print(f"Turns: {session.turn_count()}")
print(f"\nAPI format (what the LLM sees):")
for msg in session.to_api_format():
    print(f"  [{msg['role']:9s}] {msg['content'][:60]}...")

# Output:
#   Session: a3f8c2e1
#   Turns: 2
#   API format (what the LLM sees):
#     [system   ] You are a helpful AI tutor for the Netsetos course....
#     [user     ] What is RAG?...
#     [assistant] RAG stands for Retrieval-Augmented Generation......
#     [user     ] How does the retrieval part work?...

**Explanation**

This is a plain data-structure cell, not a model call - no API key needed to run it. Read it as three nested layers: a `Message` dataclass is the atom, a `ChatSession` is an append-only list of messages plus helpers, and a `SessionStore` is a dict of sessions keyed by ID. `to_api_format` is the bridge that turns your internal objects into the list-of-dicts an LLM wants.

**How the code works - step by step**
- **`Message`** - a dataclass holding `role`, `content`, an auto-filled `timestamp`, and a `tokens` count.
- **`ChatSession.__init__`** - mints a short random `session_id` and seeds the list with the system prompt if one is given.
- **`add_user_message` / `add_assistant_message`** - append a turn with the right role.
- **`to_api_format`** - strips each message down to `{"role", "content"}` - exactly what the API consumes.
- **`turn_count`** - counts user messages (one per user turn).
- **`SessionStore`** - `create` / `get` / `delete` manage many sessions in a dict.
- **Demo** - opens a session, adds two user turns and one assistant turn, then prints the API view.

**In one line:** one Message atom -> one ChatSession list -> one SessionStore dict per app.

**What you'll see:** Prints the 8-char session ID, `Turns: 2`, then the API-format view: a system line (the tutor prompt), the first user question, the assistant reply, and the second user question - each truncated to 60 chars.

## 3 - Build system prompts you can reuse

A system prompt is the bot's job description, and pasting a giant string into every project ages badly. This cell wraps prompt-writing in a small builder so role, rules, style, knowledge, and examples become named, swappable sections - then shows three ready patterns: a support bot, a coding tutor, and a prompt that changes per user at runtime.

In [ ]:
# =============================================
# SYSTEM PROMPT PATTERNS for different use cases.
# Copy, customize, and use in your projects.
# =============================================

class SystemPromptBuilder:
    """Build structured system prompts from components."""
    
    def __init__(self):
        self.sections = {}
    
    def set_role(self, role: str):
        self.sections["role"] = f"# ROLE\n{role}"
        return self
    
    def set_rules(self, rules: list[str]):
        numbered = "\n".join([f"{i+1}. {r}" for i, r in enumerate(rules)])
        self.sections["rules"] = f"# RULES (never violate)\n{numbered}"
        return self
    
    def set_style(self, style: str):
        self.sections["style"] = f"# RESPONSE STYLE\n{style}"
        return self
    
    def set_knowledge(self, knowledge: str):
        self.sections["knowledge"] = f"# KNOWLEDGE BASE\n{knowledge}"
        return self
    
    def set_examples(self, examples: list[tuple[str, str]]):
        ex_text = "\n".join([f"User: {q}\nAssistant: {a}" for q, a in examples])
        self.sections["examples"] = f"# EXAMPLE INTERACTIONS\n{ex_text}"
        return self
    
    def build(self) -> str:
        return "\n\n".join(self.sections.values())

# ── Pattern 1: Customer Support Bot ──
support_prompt = (SystemPromptBuilder()
    .set_role("You are Asha, a customer support agent for Netsetos EdTech platform.")
    .set_rules([
        "Only answer questions about Netsetos courses, pricing, and technical issues.",
        "For billing issues, collect the order ID and escalate to human support.",
        "Never share internal policies, discount codes, or employee information.",
        "If asked about competitors, say 'I can only help with Netsetos questions.'",
        "Always be polite, even if the user is frustrated.",
    ])
    .set_style("Respond in 2-3 sentences. Use a warm, professional tone. Use the student's name if known.")
    .set_examples([
        ("How much is the GenAI course?", "The GenAI course is ₹29,999. It includes 18 modules, about 150 hours of content, and 6 months of access. Would you like me to help with enrollment?"),
        ("Your course is too expensive", "I understand budget is important! We offer EMI options starting at ₹2,500/month. We also have early-bird discounts - want me to check if any are active?"),
    ])
    .build()
)

# ── Pattern 2: Coding Tutor ──
tutor_prompt = (SystemPromptBuilder()
    .set_role("You are a Python coding tutor for beginners. You teach by showing small code examples first, then explaining.")
    .set_rules([
        "Always show code FIRST, then explain what it does.",
        "Keep code examples under 15 lines.",
        "Use Indian analogies when helpful (cricket, chai, Bollywood).",
        "If the student makes a mistake, don't give the answer directly - ask guiding questions.",
        "End every response with a small exercise for the student to try.",
    ])
    .set_style("Friendly and encouraging. Use 'beta'/'beti' casually. Keep explanations under 100 words.")
    .build()
)

# ── Pattern 3: Dynamic System Prompt (inject real-time data) ──
def build_dynamic_prompt(user_name: str, user_plan: str, current_module: str) -> str:
    """System prompt that changes based on user context."""
    return (SystemPromptBuilder()
        .set_role(f"You are a learning assistant for {user_name}, currently on the {user_plan} plan.")
        .set_knowledge(f"""
Current progress: {current_module}
Plan: {user_plan} ({"full access" if user_plan == "pro" else "limited to Modules 3-5"})
Tip: If they ask about advanced topics beyond their plan, mention the upgrade option.""")
        .set_style("Personalized, encouraging. Reference their progress.")
        .build()
    )

print("Support bot prompt:")
print(support_prompt[:300] + "...\n")

print("Dynamic prompt for a specific user:")
print(build_dynamic_prompt("Rahul", "free", "Module 5: Prompt Engineering"))

# Output:
#   Support bot prompt:
#   # ROLE
#   You are Asha, a customer support agent for Netsetos EdTech platform.
#   # RULES (never violate)
#   1. Only answer questions about Netsetos courses, pricing, and technic...
#
#   Dynamic prompt for a specific user:
#   # ROLE
#   You are a learning assistant for Rahul, currently on the free plan.
#   # KNOWLEDGE BASE
#   Current progress: Module 5: Prompt Engineering
#   Plan: free (limited to Modules 3-5)

**Explanation**

This is a string-assembly cell, not a model call. `SystemPromptBuilder` is a fluent builder - every setter stashes a formatted section under a key and returns `self`, so calls chain, and `build` joins the sections with blank lines. The three patterns are just different setter combinations; the dynamic one is a function that bakes live user context into the prompt.

**How the code works - step by step**
- **`set_role` / `set_rules` / `set_style` / `set_knowledge` / `set_examples`** - each formats one titled section (numbered rules, `User:`/`Assistant:` examples) and returns `self` for chaining.
- **`build`** - joins all stored sections with double newlines into the final prompt string.
- **Pattern 1 - support bot** - chains role + five rules + style + two examples into `support_prompt`.
- **Pattern 2 - coding tutor** - role + rules + style, no examples.
- **Pattern 3 - `build_dynamic_prompt`** - a function that injects `user_name`, `user_plan`, and `current_module`, switching the knowledge text on whether the plan is `pro`.

**In one line:** name each prompt section, chain the setters, and `build` glues them together.

**What you'll see:** Prints the first 300 chars of the support-bot prompt (ROLE + start of RULES), then the full dynamic prompt for a user named Rahul on the free plan - showing the KNOWLEDGE BASE limited to Modules 3-5.

## Setup - install dependencies

The live demos below use the Google Gen AI SDK for Gemini calls and `python-dotenv` for loading keys. Uncomment this line on Colab or a fresh environment to install them.

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install google-genai python-dotenv -q  # noqa


**Explanation**

A commented-out pip line so a clean machine can pull the two runtime packages this lesson calls. It stays commented so re-running the notebook never triggers a slow reinstall.

**How the code works - step by step**
- **`!pip install google-genai python-dotenv -q`** - the Gemini SDK plus a `.env` loader; the `# noqa` marks it as an intentional shell magic.

**In one line:** uncomment once to install, then leave it commented.

**What you'll see:** (no output - the install line is commented out)

## Setup - load your API key

The memory and full-API cells make real Gemini calls, so they need a key. This cell reads it from the environment (or a `.env` file) rather than hardcoding it.

In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


**Explanation**

Environment prep, not logic. `setdefault` puts an empty `GOOGLE_API_KEY` placeholder in place only if one isn't already set, so your real key (from the shell or a `.env`) is never overwritten. Keys stay out of the source.

**How the code works - step by step**
- **`import os`** - access to the process environment.
- **`os.environ.setdefault("GOOGLE_API_KEY", "")`** - reserves the slot without clobbering an existing value; get a free key at aistudio.google.com.

**In one line:** any one provider key in the environment is enough to start.

**What you'll see:** (no output - environment prep). Cells that call Gemini will fail with an auth error until a real key is present.

## 4 - Keep the conversation inside a token budget

Histories grow forever, but context windows and bills don't. This cell builds a `MemoryManager` with three ways to shrink history before you send it: a sliding window (drop the oldest), summarize-and-keep (compress the old, keep the recent verbatim), and entity memory (distill just the key facts).

> **Needs a Google API key** (set GOOGLE_API_KEY) - strategies 2 and 3 make live Gemini calls to summarize/extract.

In [ ]:
# =============================================
# CONVERSATION MEMORY MANAGER  (google-genai SDK)
# 3 strategies for keeping history within budget.
# pip install google-genai
# =============================================

from google import genai

client = genai.Client()      # reads GOOGLE_API_KEY from the environment
MODEL = "gemini-2.5-flash"   # gemini-2.0-flash was shut down 2026-06-01

def rough_tokens(text: str) -> int:
    # Local estimate (~4 chars/token for English): free, instant, good enough
    # for TRIM decisions. Bill from response.usage_metadata (the 7.3 rule) -
    # never from an estimator.
    return max(1, len(text) // 4)

class MemoryManager:
    """Manage conversation memory within token budgets."""

    def __init__(self, max_history_tokens: int = 8000):
        self.max_tokens = max_history_tokens

    def _count(self, messages: list[dict]) -> int:
        return sum(rough_tokens(m["content"]) for m in messages)

    def needs_trimming(self, messages: list[dict]) -> bool:
        return self._count(messages) > self.max_tokens

    # -- Strategy 1: Sliding Window --
    def sliding_window(self, messages: list[dict]) -> list[dict]:
        """Keep system prompt + as many recent messages as fit."""
        system = [m for m in messages if m["role"] == "system"]
        history = [m for m in messages if m["role"] != "system"]
        budget = self.max_tokens - self._count(system)
        kept: list[dict] = []
        for msg in reversed(history):            # newest first
            cost = rough_tokens(msg["content"])
            if budget < cost:
                break
            kept.insert(0, msg)
            budget -= cost
        return system + kept

    # -- Strategy 2: Summarize + Keep Recent --
    def summarize_and_keep(self, messages: list[dict], keep_recent: int = 6) -> list[dict]:
        """Compress old messages into a summary; keep recent ones verbatim."""
        system = [m for m in messages if m["role"] == "system"]
        history = [m for m in messages if m["role"] != "system"]
        if len(history) <= keep_recent:
            return messages
        old, recent = history[:-keep_recent], history[-keep_recent:]
        old_text = "\n".join(f"{m['role']}: {m['content']}" for m in old)
        resp = client.models.generate_content(
            model=MODEL,
            contents="Summarize this conversation in 3-4 bullet points. "
                     "Keep key decisions, facts, and user preferences:\n\n" + old_text,
        )
        summary = {"role": "system", "content": f"[Earlier conversation summary]\n{resp.text}"}
        # TRAP - we shipped this bug: if the call site later filters messages
        # to user/assistant roles only, this summary is silently DISCARDED and
        # you pay for a summarization call the model never sees. Step 10 merges
        # every system-role message into system_instruction. Watch for it.
        return system + [summary] + recent

    # -- Strategy 3: Entity Memory (remember key facts) --
    def extract_entities(self, messages: list[dict]) -> str:
        """Distill key facts (the index card taped inside the notebook cover)."""
        text = "\n".join(f"{m['role']}: {m['content']}"
                         for m in messages if m["role"] != "system")
        resp = client.models.generate_content(
            model=MODEL,
            contents="Extract key facts as bullet points - user's name, goal, "
                     "decisions made, preferences, important numbers/dates:\n\n" + text,
        )
        return resp.text.strip()

    def trim(self, messages: list[dict], strategy: str = "summarize") -> list[dict]:
        if not self.needs_trimming(messages):
            return messages
        if strategy == "window":
            return self.sliding_window(messages)
        if strategy == "summarize":
            return self.summarize_and_keep(messages)
        raise ValueError(f"unknown strategy: {strategy}")   # fail loud, not silent

# -- Demo --
memory = MemoryManager(max_history_tokens=2000)
long_chat = [{"role": "system", "content": "You are a helpful tutor."}]
topics = ["RAG basics", "embeddings", "chunking", "vector DB", "retrieval",
          "reranking", "generation", "evaluation", "deployment", "monitoring"]
for t in topics:
    long_chat.append({"role": "user", "content": f"Explain {t} in detail with code examples."})
    long_chat.append({"role": "assistant", "content": f"{t} involves several important concepts..." * 10})

print(f"Original: {len(long_chat)} messages, ~{memory._count(long_chat)} tokens")
trimmed = memory.trim(long_chat, strategy="summarize")
print(f"Trimmed:  {len(trimmed)} messages, ~{memory._count(trimmed)} tokens")

# Output:
#   Original: 21 messages, ~2860 tokens
#   Trimmed:  9 messages, ~1218 tokens
#   (1 system prompt + 1 summary + last 6 messages; the summary cost one
#    cheap gemini-2.5-flash call - meter it like any other call, 7.3 rule)

**Explanation**

This is a history-compaction toolkit around a cheap local token estimate. `rough_tokens` counts ~4 chars per token - good enough to DECIDE when to trim, never to bill (bill from `usage_metadata`, the 7.3 rule). The three strategies trade fidelity for cost differently, and `trim` is the dispatcher that fails loud on an unknown strategy name.

**How the code works - step by step**
- **`rough_tokens`** - a free, instant length/4 estimate for trim decisions only.
- **`_count` / `needs_trimming`** - sum estimated tokens and compare to `max_tokens`.
- **`sliding_window`** - always keep the system prompt, then add newest-first until the budget runs out.
- **`summarize_and_keep`** - split history into old + last `keep_recent`, send the old block to Gemini for a 3-4 bullet summary, and prepend that as a system message (with a TRAP note: a later role filter can silently drop it).
- **`extract_entities`** - ask Gemini to pull names, goals, decisions, and key numbers as bullets.
- **`trim`** - routes to `window` or `summarize`; raises `ValueError` on anything else.
- **Demo** - builds a 21-message chat over 10 RAG topics and trims with the summarize strategy.

**In one line:** estimate the size, pick a shrink strategy, keep the system prompt sacred.

**What you'll see:** Prints `Original: 21 messages, ~2860 tokens` then `Trimmed: 9 messages, ~1218 tokens` - one system prompt + one Gemini-generated summary + the last 6 messages. The summary costs one cheap flash call.

## 5 - Make sessions survive a restart with Redis

An in-memory `SessionStore` vanishes when the server restarts. This cell moves sessions into Redis with a TTL so they persist across restarts and auto-expire when stale - using the async client that matches async FastAPI.

> **Needs a running Redis server** (redis://localhost:6379) to actually execute.

In [ ]:
# =============================================
# PERSISTENT SESSIONS: Redis (async client)
# Sessions survive server restarts.  pip install redis
# =============================================

import json
import redis.asyncio as redis          # async client - matches async FastAPI
from datetime import datetime

class PersistentSessionStore:
    """Chat sessions stored in Redis - survive restarts."""

    def __init__(self, redis_url: str = "redis://localhost:6379", ttl_hours: int = 24):
        self.redis = redis.from_url(redis_url, decode_responses=True)
        self.ttl = ttl_hours * 3600

    def _key(self, session_id: str) -> str:
        return f"chat:session:{session_id}"

    async def save(self, session: ChatSession) -> None:
        data = {
            "session_id": session.session_id,
            "messages": [{"role": m.role, "content": m.content, "ts": m.timestamp}
                         for m in session.messages],
            "created": session.created_at.isoformat(),
            "updated": datetime.now().isoformat(),
        }
        await self.redis.setex(self._key(session.session_id), self.ttl, json.dumps(data))

    async def load(self, session_id: str) -> ChatSession | None:
        raw = await self.redis.get(self._key(session_id))
        if raw is None:
            return None
        data = json.loads(raw)
        session = ChatSession(session_id=data["session_id"])
        session.created_at = datetime.fromisoformat(data["created"])   # restore, don't reset
        for m in data["messages"]:
            session.messages.append(Message(role=m["role"], content=m["content"],
                                            timestamp=m.get("ts", 0)))
        return session

    async def list_sessions(self) -> list[str]:
        # SCAN, never KEYS: KEYS walks every key in one blocking O(N) sweep -
        # it freezes the whole Redis server in production. SCAN iterates in
        # small batches the server can breathe between.
        ids: list[str] = []
        async for key in self.redis.scan_iter(match="chat:session:*"):
            ids.append(key.split(":")[-1])
        return ids

# Why Redis for chat sessions?
#   - Fast: typically sub-millisecond reads (in-memory, RAM-speed)
#   - TTL: sessions auto-expire (the filing cabinet's auto-shred date)
#   - Persistent: survives restarts with AOF/RDB snapshots
# For PERMANENT history (analytics, compliance): also archive to PostgreSQL
# asynchronously. Redis = hot path, database = basement archive.

# Output: (demo with a local Redis, then a simulated restart)
#   saved  chat:session:a3f8c2e1  ttl=86400s
#   -- server restarted --
#   loaded a3f8c2e1: 3 messages, created 2026-07-02T14:03:11


**Explanation**

This is a serialization layer, not a model call: it turns a `ChatSession` into JSON, stores it under a namespaced key with an expiry, and rebuilds the object on load. The key detail is `scan_iter` instead of `KEYS` - listing sessions must never block the whole Redis server with an O(N) sweep.

**How the code works - step by step**
- **`__init__`** - opens an async Redis connection and stores a TTL in seconds.
- **`_key`** - namespaces every session as `chat:session:{id}`.
- **`save`** - packs id, messages, and timestamps into a dict and writes it with `setex` (value + expiry in one call).
- **`load`** - reads the JSON back, rebuilds the `ChatSession`, and restores the original `created_at` instead of resetting it.
- **`list_sessions`** - iterates keys with `scan_iter` in small batches, never the blocking `KEYS`.

**In one line:** JSON-in with a TTL, JSON-out on load, SCAN (never KEYS) to enumerate.

**What you'll see:** Comments show the intended run against a local Redis: `saved chat:session:a3f8c2e1 ttl=86400s`, a simulated restart, then `loaded a3f8c2e1: 3 messages` with the original creation time preserved. Without a live Redis it won't run.

## 6 - Build the LLM client once, inject it everywhere

Creating a fresh Gemini client on every request pays a TCP+TLS handshake tax each time. This cell shows the two FastAPI patterns that fix it: a lifespan that builds one client at startup, and dependency injection (`Depends`) that hands that singleton to any route that asks.

> **FastAPI app** - run with `uvicorn minimal_di:app`; shown here as reference, not executed in the notebook.

In [ ]:
# minimal_di.py - the two patterns that carry every FastAPI AI service
from contextlib import asynccontextmanager
from typing import Annotated
from fastapi import Depends, FastAPI, Request

@asynccontextmanager
async def lifespan(app: FastAPI):
    from google import genai
    app.state.llm = genai.Client()      # created ONCE at startup
    print("startup: LLM client ready")
    yield
    print("shutdown: flush logs, close pools here")

app = FastAPI(lifespan=lifespan)

def get_llm(request: Request):
    return request.app.state.llm        # the singleton, injected

LLMClient = Annotated[object, Depends(get_llm)]   # clean reusable alias

@app.get("/models")
async def list_models(llm: LLMClient):
    # this route never BUILT a client - it was delivered to the station
    return {"client": type(llm).__name__}

# Output: uvicorn minimal_di:app
#   startup: LLM client ready
#   GET /models  ->  {"client": "Client"}
#   ^C  shutdown: flush logs, close pools here


**Explanation**

This is a wiring pattern, not runnable notebook output. `lifespan` is an async context manager: everything before `yield` runs once at startup (build the client), everything after runs at shutdown (cleanup). `get_llm` reaches into `app.state` to return that one client, and the `LLMClient` type alias makes injecting it a one-word annotation.

**How the code works - step by step**
- **`lifespan`** - builds `app.state.llm` once at startup, prints a ready message, and reserves the post-`yield` block for shutdown cleanup.
- **`app = FastAPI(lifespan=lifespan)`** - registers that startup/shutdown hook.
- **`get_llm`** - a dependency that returns the singleton off `request.app.state`.
- **`LLMClient = Annotated[..., Depends(get_llm)]`** - a reusable alias so routes just type `llm: LLMClient`.
- **`/models` route** - never builds a client; it receives the injected one and reports its type.

**In one line:** one client at startup, injected on demand - no route ever constructs its own.

**What you'll see:** Comments show the reference run under uvicorn: `startup: LLM client ready`, `GET /models -> {"client": "Client"}`, and a shutdown line on Ctrl-C. Nothing prints in the notebook itself.

## 7 - Stream tokens to the browser with SSE

Users hate staring at a spinner while a full answer generates. Server-Sent Events (SSE) push each token to the browser as it arrives. This cell uses FastAPI's native SSE support (since 0.135) to stream a fake token list, with a disconnect check so you stop paying the moment the user closes the tab.

> **FastAPI app** - run with uvicorn and hit it via `curl -N`; shown here as reference.

In [ ]:
# sse_minimal.py - native SSE since FastAPI 0.135
# pip install "fastapi>=0.135" uvicorn
import asyncio, json
from fastapi import FastAPI, Request
from fastapi.sse import EventSourceResponse, ServerSentEvent

app = FastAPI()

@app.post("/chat/stream")
async def stream(request: Request):
    async def gen():
        tokens = ["The", " illusion", " of", " memory", " is", " engineered."]
        for token in tokens:                    # stand-in for the LLM stream
            if await request.is_disconnected():
                return                          # tab closed: stop paying
            yield ServerSentEvent(data=json.dumps({"t": token}))
            await asyncio.sleep(0.2)
        yield ServerSentEvent(data="[DONE]")
    return EventSourceResponse(gen())

# Output: curl -N -X POST localhost:8000/chat/stream
#   data: {"t": "The"}
#
#   data: {"t": " illusion"}
#   ...
#   data: [DONE]
#   (every frame ends with a blank line - the \n\n terminator from 7.2)


**Explanation**

This is a streaming-endpoint pattern. An async generator yields one `ServerSentEvent` per token, and `EventSourceResponse` wraps that generator into a proper SSE stream. The token list stands in for a real LLM stream; the important production move is `request.is_disconnected()`, which aborts generation when the client is gone.

**How the code works - step by step**
- **`gen`** - an async generator over a fixed token list (the LLM-stream stand-in).
- **`request.is_disconnected()`** - checked each loop; returns early if the tab closed, so you stop generating and stop paying.
- **`yield ServerSentEvent(...)`** - emits each token as a JSON data frame, with a 0.2s pause to simulate generation latency.
- **`[DONE]` sentinel** - a final frame signaling the stream is complete.
- **`EventSourceResponse(gen())`** - turns the generator into the SSE HTTP response.

**In one line:** yield one SSE frame per token, bail out the instant the client disconnects.

**What you'll see:** Comments show the `curl -N` output: `data: {"t": "The"}`, `data: {"t": " illusion"}`, ... ending in `data: [DONE]`, each frame closed by a blank line (the `\n\n` SSE terminator). Not executed in the notebook.

## 8 - One queryable JSON log line per request

At 3 AM you can't grep prose. This cell configures `structlog` to emit structured JSON and adds middleware that stamps a short `request_id` onto every log line for that request - so you can trace one user's whole flow, and return the ID in a response header.

> **FastAPI app** - reference wiring; run under uvicorn to see real log lines.

In [ ]:
# observability.py - structured logs with per-request context
# pip install structlog
import time, uuid, structlog
from fastapi import FastAPI, Request

structlog.configure(processors=[
    structlog.contextvars.merge_contextvars,   # carries request_id everywhere
    structlog.processors.add_log_level,
    structlog.processors.TimeStamper(fmt="iso"),
    structlog.processors.JSONRenderer(),
])
logger = structlog.get_logger()
app = FastAPI()

@app.middleware("http")
async def trace_requests(request: Request, call_next):
    rid = str(uuid.uuid4())[:8]
    structlog.contextvars.bind_contextvars(request_id=rid)   # sticks to every
    t0 = time.perf_counter()                                 # log line below
    response = await call_next(request)
    logger.info("request_done", path=request.url.path,
                ms=round((time.perf_counter() - t0) * 1000, 1))
    response.headers["X-Request-ID"] = rid    # the flight number, returned
    return response

# After every LLM call, log the numbers that answer 3 AM questions:
#   logger.info("llm_call", model=MODEL, ms=latency_ms,
#               tokens_in=usage.prompt_token_count,
#               tokens_out=usage.candidates_token_count)

# Output: one queryable JSON line per event
#   {"event": "llm_call", "request_id": "a3f8c2e1",
#    "model": "gemini-2.5-flash", "ms": 842.3,
#    "tokens_in": 412, "tokens_out": 58,
#    "level": "info", "timestamp": "2026-07-02T14:03:11Z"}


**Explanation**

This is an observability setup, not a model call. The processor chain ends in `JSONRenderer`, so every log event becomes one machine-parseable JSON line. `merge_contextvars` is the trick: bind `request_id` once in the middleware and it silently attaches to every log line emitted while that request runs.

**How the code works - step by step**
- **`structlog.configure`** - a processor pipeline: merge context vars, add level, add an ISO timestamp, render as JSON.
- **`trace_requests` middleware** - mints an 8-char request ID, binds it with `bind_contextvars`, starts a timer, runs the request.
- **`logger.info("request_done", ...)`** - logs the path and elapsed milliseconds after the response.
- **`X-Request-ID` header** - returns the same ID to the caller (the flight number).
- **The commented `llm_call` log** - the template for logging model, latency, and in/out token counts after every LLM call.

**In one line:** bind an ID per request, and every JSON log line carries it automatically.

**What you'll see:** Comments show one example JSON line for an `llm_call` event - with `request_id`, `model`, `ms`, `tokens_in`, `tokens_out`, level, and timestamp. Nothing prints in the notebook.

## 9 - Package it: multi-stage Docker + Cloud Run

To ship the service you need a small, reproducible image. This cell writes a multi-stage `Dockerfile` - a builder stage installs dependencies with `uv`, and a slim runtime stage copies only the installed packages - then shows the Cloud Run deploy command that injects secrets at deploy time.

In [ ]:
%%writefile Dockerfile
# Dockerfile - multi-stage: pack the furniture, leave the workshop
FROM python:3.12-slim AS builder
COPY --from=ghcr.io/astral-sh/uv:latest /uv /usr/local/bin/uv
WORKDIR /app
COPY requirements.txt .
RUN uv pip install --system --no-cache -r requirements.txt

FROM python:3.12-slim
WORKDIR /app
COPY --from=builder /usr/local/lib/python3.12/site-packages /usr/local/lib/python3.12/site-packages
COPY --from=builder /usr/local/bin/uvicorn /usr/local/bin/uvicorn
COPY app/ app/
# exec-form CMD: uvicorn receives SIGTERM directly -> graceful shutdown.
# Shell form (CMD uvicorn ...) wraps it in /bin/sh, which eats the signal.
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8080"]

# Deploy - secrets injected at deploy time, never baked into the image:
#   gcloud run deploy chat-api --source . --region asia-south1 \
#     --set-secrets="GOOGLE_API_KEY=google-key:latest" \
#     --timeout 3600 --min-instances 1
# Output:
#   Building using Dockerfile and deploying container to Cloud Run...
#   Service [chat-api] revision [chat-api-00001-abc] has been deployed
#   Service URL: https://chat-api-xxxxx.asia-south1.run.app


**Explanation**

The `%%writefile` magic saves the cell body to a `Dockerfile` on disk rather than running Python. Multi-stage means "pack the furniture in one room, carry only the boxes to the new house": build tooling stays in the builder image, the final image ships just the runtime and site-packages. The exec-form `CMD` is deliberate so uvicorn receives SIGTERM directly and shuts down gracefully.

**How the code works - step by step**
- **`FROM python:3.12-slim AS builder`** - the build stage; copies in the `uv` binary and installs `requirements.txt` system-wide.
- **Second `FROM python:3.12-slim`** - a fresh runtime stage that copies only the installed site-packages and the uvicorn binary from the builder.
- **`COPY app/ app/`** - adds your application code.
- **exec-form `CMD ["uvicorn", ...]`** - a JSON-array command so uvicorn is PID 1 and gets SIGTERM directly (shell form would swallow the signal).
- **The deploy comment** - `gcloud run deploy` with `--set-secrets` so the API key is mounted at deploy time, never baked into the image.

**In one line:** build fat, ship slim, and let uvicorn own the shutdown signal.

**What you'll see:** The `%%writefile` magic prints `Writing Dockerfile` (or `Overwriting` on a re-run). The `gcloud` build/deploy lines are comments showing the expected Cloud Run success output and service URL.

## 10 - Test the whole thing with a fake LLM

You can't test a chat API by calling the real model on every CI run - it's slow, flaky, and it bills you. This cell runs the FastAPI app in-process with `AsyncClient` and swaps in a fake LLM (`AsyncMock`), so a full create-session-then-chat round-trip runs offline in a fraction of a second.

> **Reference test** - run with `pytest`; needs the `project_chat_api` module and pytest installed.

In [ ]:
# test_chat.py - the fire drill with a FAKE fire (no live API, no bill)
# pip install pytest pytest-asyncio httpx
from unittest.mock import AsyncMock
import pytest
from httpx import ASGITransport, AsyncClient
from project_chat_api import app

def fake_llm_response(text="Namaste! How can I help?"):
    resp = AsyncMock()
    resp.text = text
    resp.usage_metadata.prompt_token_count = 42
    resp.usage_metadata.candidates_token_count = 12
    return resp

@pytest.mark.asyncio
async def test_chat_turn():
    # ASGITransport skips lifespan - install the fake singleton yourself
    app.state.llm = AsyncMock()
    app.state.llm.aio.models.generate_content = AsyncMock(
        return_value=fake_llm_response())
    async with AsyncClient(transport=ASGITransport(app=app),
                           base_url="http://test") as http:
        r = await http.post("/sessions", json={})
        sid = r.json()["session_id"]
        r = await http.post(f"/sessions/{sid}/chat",
                            json={"message": "Hello!"})
    assert r.status_code == 200
    assert r.json()["turn"] == 1

# Also test the error paths - that's where production failures live:
#   mock raising a timeout  -> expect 503 AND the user turn rolled back
#   provider returns a 429  -> expect your handler's Retry-After header

# Output: pytest -q
#   .                                                        [100%]
#   1 passed in 0.31s


**Explanation**

This is a test harness, not app code - a fire drill with a fake fire. `ASGITransport` drives the app directly without a network socket (but it skips lifespan, so you install the fake client onto `app.state` yourself). The fake `generate_content` returns a canned response object with the token fields the route reads, so the assertions exercise your routing and turn logic, not Gemini.

**How the code works - step by step**
- **`fake_llm_response`** - builds an `AsyncMock` whose `.text` and `usage_metadata` token counts mimic a real Gemini reply.
- **`test_chat_turn`** - installs a fake `app.state.llm` (lifespan is skipped under ASGITransport) whose async `generate_content` returns that canned response.
- **`AsyncClient(transport=ASGITransport(app=app))`** - talks to the app in-process, no live server.
- **The round-trip** - `POST /sessions` to get a session ID, then `POST /sessions/{id}/chat`.
- **Assertions** - status 200 and `turn == 1`.
- **Error-path note** - also test a timeout (expect 503 and the user turn rolled back) and a 429 (expect a Retry-After header).

**In one line:** run the app in-memory with a mocked model, then assert the routing and turn count.

**What you'll see:** Comments show the `pytest -q` result: a single dot and `1 passed in 0.31s`. It won't run in the notebook without the `project_chat_api` module and a pytest session.

## 11 - The full production chat API

Everything so far snaps together here. This is the honest, end-to-end service: it reuses your session manager, prompt builder, and memory manager, then adds an async Gemini client, CORS, request validation, history trimming that PERSISTS its compaction, a blocking chat route, an SSE streaming route, and error roll-back.

> **Needs a Google API key** (set GOOGLE_API_KEY) and runs under `uvicorn project_chat_api:app --reload` - shown here as the capstone reference.

In [ ]:
# =============================================
# PRODUCTION CHAT API - the honest version
# Sessions + Memory + Streaming + Persistence
# pip install "fastapi>=0.135" uvicorn google-genai
# Reuses the classes you built in steps 1-3:
from part1_session_manager import Message, SessionStore
from part2_system_prompts import SystemPromptBuilder
from part3_memory_strategies import MemoryManager
# =============================================

import asyncio, json
from contextlib import asynccontextmanager

from fastapi import FastAPI, HTTPException, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.sse import EventSourceResponse, ServerSentEvent   # FastAPI 0.135+
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

@asynccontextmanager
async def lifespan(app: FastAPI):
    app.state.llm = genai.Client()   # ONE client for the app's lifetime -
    yield                            # a per-request client pays a TCP+TLS
                                     # handshake tax on every single call

app = FastAPI(title="Netsetos Chat API", lifespan=lifespan)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["https://app.example.com"],   # never "*" in production
    allow_methods=["GET", "POST", "DELETE"],
    allow_headers=["*"],
)

MODEL = "gemini-2.5-flash"
sessions = SessionStore()
memory = MemoryManager(max_history_tokens=8000)

DEFAULT_SYSTEM = (SystemPromptBuilder()
    .set_role("You are a helpful AI tutor for the Netsetos GenAI course.")
    .set_rules(["Keep answers under 150 words.", "Use code examples when helpful."])
    .build())

class CreateSession(BaseModel):
    # Client-supplied system prompts are an injection surface (step 9):
    # cap the size; in production prefer an allow-list of named templates.
    system_prompt: str = Field(default=DEFAULT_SYSTEM, max_length=4000)

class ChatMessage(BaseModel):
    message: str = Field(min_length=1, max_length=8000)

def prepare_context(session) -> tuple[str, list[types.Content]]:
    """Trim history, PERSIST the compaction, split system text from turns."""
    api_messages = memory.trim(session.to_api_format(), strategy="summarize")
    if len(api_messages) < len(session.messages):
        # Persist the compaction - otherwise next turn re-summarizes the
        # ever-growing ORIGINAL: one wasted LLM call per turn, forever.
        session.messages = [Message(role=m["role"], content=m["content"])
                            for m in api_messages]
    # EVERY system-role message (original prompt + any summary) reaches the
    # model. This line is the fix for the silently-discarded-summary bug.
    system_text = "\n\n".join(m["content"] for m in api_messages
                               if m["role"] == "system")
    contents = [types.Content(role="user" if m["role"] == "user" else "model",
                              parts=[types.Part(text=m["content"])])
                for m in api_messages if m["role"] in ("user", "assistant")]
    return system_text, contents

@app.post("/sessions")
async def create_session(req: CreateSession):
    session = sessions.create(system_prompt=req.system_prompt)
    return {"session_id": session.session_id, "turns": 0}

@app.get("/sessions/{session_id}")
async def get_session(session_id: str):
    session = sessions.get(session_id)
    if not session:
        raise HTTPException(404, "Session not found")
    return {"session_id": session.session_id, "turns": session.turn_count(),
            "messages": session.to_api_format()}

@app.post("/sessions/{session_id}/chat")
async def chat(session_id: str, req: ChatMessage):
    session = sessions.get(session_id)
    if not session:
        raise HTTPException(404, "Session not found")
    session.add_user_message(req.message)
    try:
        # prepare_context may call the LLM to summarize (sync helper from
        # step 3) - to_thread keeps it OFF the event loop. The chat call
        # itself uses the native async client: client.aio.
        system_text, contents = await asyncio.to_thread(prepare_context, session)
        resp = await app.state.llm.aio.models.generate_content(
            model=MODEL, contents=contents,
            config=types.GenerateContentConfig(system_instruction=system_text))
    except Exception:
        session.messages.pop()   # roll back the user turn - no orphan on retry
        raise HTTPException(503, "Model call failed - please retry")
    session.add_assistant_message(resp.text)
    return {"response": resp.text, "turn": session.turn_count(),
            "tokens": {"in": resp.usage_metadata.prompt_token_count,
                       "out": resp.usage_metadata.candidates_token_count}}

@app.post("/sessions/{session_id}/chat/stream")
async def chat_stream(session_id: str, req: ChatMessage, request: Request):
    session = sessions.get(session_id)
    if not session:
        raise HTTPException(404, "Session not found")
    session.add_user_message(req.message)
    system_text, contents = await asyncio.to_thread(prepare_context, session)

    async def generate():
        full = ""
        stream = await app.state.llm.aio.models.generate_content_stream(
            model=MODEL, contents=contents,   # FULL trimmed history - the
            config=types.GenerateContentConfig(   # goldfish bug fix
                system_instruction=system_text))
        async for chunk in stream:
            if await request.is_disconnected():   # user closed the tab:
                break                             # stop generating, stop paying
            if chunk.text:
                full += chunk.text
                yield ServerSentEvent(data=json.dumps(
                    {"type": "token", "content": chunk.text}))
        if full:
            session.add_assistant_message(full)   # save even a partial answer -
        yield ServerSentEvent(data=json.dumps(     # history stays consistent
            {"type": "done", "turn": session.turn_count()}))

    return EventSourceResponse(generate())

@app.delete("/sessions/{session_id}")
async def delete_session(session_id: str):
    sessions.delete(session_id)
    return {"deleted": session_id}

# Run: uvicorn project_chat_api:app --reload
# Output: (turn 3 of a conversation - note the history-aware answer and
#          the input-token count carrying the trimmed history)
#   POST /sessions/{id}/chat  {"message": "So which one should I pick?"}
#   {"response": "For a cracked screen, choose the replacement - it ships
#     within 5 days and your order #4521 qualifies...",
#    "turn": 3, "tokens": {"in": 412, "out": 58}}
#   -- the cold-open demo would have answered: "Which one of WHAT?"


**Explanation**

This cell wires the whole system into one FastAPI app. The star is `prepare_context`: it trims history, writes the trimmed version back onto the session (so you never re-summarize the ever-growing original), and splits every system-role message into `system_instruction` while the user/assistant turns become `types.Content` - the exact fix for the silently-discarded-summary trap from Step 4. Both chat routes send the FULL trimmed history, which is the cure for the cold-open goldfish bug.

**How the code works - step by step**
- **Reuse imports** - pulls `Message`/`SessionStore`, `SystemPromptBuilder`, and `MemoryManager` from steps 2-4.
- **`lifespan`** - builds one `genai.Client` for the app's whole lifetime (a per-request client pays a handshake tax).
- **CORS middleware** - allow-lists a specific origin and methods, never `*` in production.
- **`CreateSession` / `ChatMessage`** - Pydantic models that cap prompt and message length (client system prompts are an injection surface).
- **`prepare_context`** - trims via the summarize strategy, persists the compaction back onto the session, then splits system text from user/model `Content` turns.
- **`POST /sessions`** - creates a session and returns its ID.
- **`GET /sessions/{id}`** - returns turn count and full message history, 404 if missing.
- **`POST /sessions/{id}/chat`** - adds the user turn, runs `prepare_context` via `asyncio.to_thread` (its summarize call is sync), awaits the native async `client.aio` call, and on any error pops the user turn (no orphaned message on retry) and raises 503.
- **`POST /sessions/{id}/chat/stream`** - the same context prep, then `generate_content_stream` yielding SSE token frames, stopping on disconnect and saving even a partial answer.
- **`DELETE /sessions/{id}`** - drops a session.

**In one line:** trim + persist + split system-from-turns, call the async client, roll back on failure.

**What you'll see:** Comments show turn 3 of a live conversation: `POST /sessions/{id}/chat` with "So which one should I pick?" returns a history-aware answer that knows it's about order #4521's cracked screen, with `"turn": 3` and `tokens: {"in": 412, "out": 58}` - versus the cold-open bot that would have said "Which one of WHAT?"

You started from one line of backend code that shipped a bot with amnesia, and you ended with a chat service that remembers. The pieces stack in a clear order: a session store holds each user's history, a system-prompt builder shapes the bot's behavior, a memory manager keeps that history inside a token budget, Redis makes it survive restarts, and FastAPI wraps it all with dependency injection, SSE streaming, structured logs, a Docker image, and tests. Carry the full production API from Step 11 forward - the async client, the roll-back-on-error pattern, and the system-instruction split are the exact moves you reuse in Module 11 when these chat services grow into agents.